# 01 — Bulk data (GSE39582)

Download and preprocess **GSE39582** (CRC bulk RNA-seq, MSI/MSS). Saves arrays under `DATA_DIR` for downstream notebooks.

**Colab:** set `USE_DRIVE = True` and mount Google Drive, or keep `False` and use `/content/data`.


In [1]:
# Optional: Google Colab
from pathlib import Path

USE_DRIVE = False  # set True on Colab to persist on Drive
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/BulkCellGNN_data")
else:
    DATA_DIR = Path.cwd() / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_DIR =", DATA_DIR.resolve())


DATA_DIR = C:\Users\krith\OneDrive - The University of Memphis\Neural Network\Project\data


In [2]:
# Install (Colab or local)
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "GEOparse", "pandas", "numpy", "scikit-learn", "matplotlib"])


0

In [3]:
import os, random
from pathlib import Path

SEED = 42
random.seed(SEED)
os.environ.setdefault("PYTHONHASHSEED", str(SEED))
import numpy as np
np.random.seed(SEED)

import pandas as pd
import GEOparse


In [4]:
# Load GSE39582 (downloads / caches under destdir)
GEO_CACHE = DATA_DIR / "geo_cache"
GEO_CACHE.mkdir(parents=True, exist_ok=True)
gse = GEOparse.get_GEO("GSE39582", destdir=str(GEO_CACHE))
print("Loaded", gse.name, "| GSMs:", len(gse.gsms))


24-Apr-2026 20:32:50 DEBUG utils - Directory C:\Users\krith\OneDrive - The University of Memphis\Neural Network\Project\data\geo_cache already exists. Skipping.
24-Apr-2026 20:32:50 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE39nnn/GSE39582/soft/GSE39582_family.soft.gz to C:\Users\krith\OneDrive - The University of Memphis\Neural Network\Project\data\geo_cache\GSE39582_family.soft.gz
100%|███████████████████████████████████████████████████████████████████████████████| 274M/274M [01:12<00:00, 3.96MB/s]
24-Apr-2026 20:34:03 DEBUG downloader - Size validation passed
24-Apr-2026 20:34:03 DEBUG downloader - Moving C:\Users\krith\AppData\Local\Temp\tmpm9y9d9wo to C:\Users\krith\OneDrive - The University of Memphis\Neural Network\Project\data\geo_cache\GSE39582_family.soft.gz
24-Apr-2026 20:34:03 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE39nnn/GSE39582/soft/GSE39582_family.soft.gz
24-Apr-2026 20:34:03 INFO GEOparse - Parsing C

Loaded GSE39582 | GSMs: 585


In [5]:
# Build sample x gene expression matrix from GSM tables
# GEOparse stores per-GSM tables; we align on common gene index (ID_REF / NAME / Gene Symbol depending on GPL)

def _gsm_frame(gsm):
    t = gsm.table.copy()
    return t

frames = []
gsm_order = []
for gsm_name in sorted(gse.gsms.keys()):
    gsm = gse.gsms[gsm_name]
    t = _gsm_frame(gsm)
    if t.empty:
        continue
    # common column names across GEO tables
    if "VALUE" not in t.columns:
        continue
    gene_col = None
    for c in ("Gene Symbol", "GENE_SYMBOL", "NAME", "ID_REF"):
        if c in t.columns:
            gene_col = c
            break
    if gene_col is None:
        gene_col = t.columns[0]
    s = t.groupby(gene_col)["VALUE"].mean().astype(float)
    s.name = gsm_name
    frames.append(s)
    gsm_order.append(gsm_name)

if not frames:
    raise RuntimeError("No GSM expression tables found. Inspect gse.gsms[...].table columns manually.")

expr_wide = pd.concat(frames, axis=1).T  # samples x genes (may contain NaN for missing)
expr_wide = expr_wide.loc[:, expr_wide.columns.notna()]
expr_wide.columns = [str(c) for c in expr_wide.columns]
expr_wide = expr_wide.dropna(axis=1, how="all")
print("expr_wide shape (pre-clean):", expr_wide.shape)


expr_wide shape (pre-clean): (585, 54675)


In [8]:
# verifying labels
print("phenotype_data shape:", gse.phenotype_data.shape)
print("\nColumn names:")
print(list(gse.phenotype_data.columns))

print("\nFirst sample full row:")
first_sid = list(gse.phenotype_data.index)[0]
print(f"Sample ID: {first_sid}")
for col, val in gse.phenotype_data.loc[first_sid].items():
    print(f"  {col}: {val}")

phenotype_data shape: (585, 66)

Column names:
['title', 'geo_accession', 'status', 'submission_date', 'last_update_date', 'type', 'channel_count', 'source_name_ch1', 'organism_ch1', 'taxid_ch1', 'characteristics_ch1.0.organism', 'characteristics_ch1.1.dataset', 'characteristics_ch1.2.Sex', 'characteristics_ch1.3.age.at.diagnosis (year)', 'characteristics_ch1.4.tnm.stage', 'characteristics_ch1.5.tnm.t', 'characteristics_ch1.6.tnm.n', 'characteristics_ch1.7.tnm.m', 'characteristics_ch1.8.tumor.location', 'characteristics_ch1.9.chemotherapy.adjuvant', 'characteristics_ch1.10.chemotherapy.adjuvant.type', 'characteristics_ch1.11.rfs.event', 'characteristics_ch1.12.rfs.delay', 'characteristics_ch1.13.os.event', 'characteristics_ch1.14.os.delay (months)', 'characteristics_ch1.15.mmr.status', 'characteristics_ch1.16.cimp.status', 'characteristics_ch1.17.cin.status', 'characteristics_ch1.18.tp53.mutation', 'characteristics_ch1.19.tp53.mutation.dna', 'characteristics_ch1.20.tp53.mutation.exon.n

In [9]:
# MSI / MSS labels from phenotype metadata
# GSE39582 uses MMR status: pMMR = MSS (0), dMMR = MSI (1)

MMR_COL = "characteristics_ch1.15.mmr.status"

labels = []
keep_samples = []

for sid in expr_wide.index:
    if sid not in gse.phenotype_data.index:
        continue
    val = str(gse.phenotype_data.loc[sid, MMR_COL]).strip().lower()
    if val == "pmmr":
        lab = 0   # MSS
    elif val == "dmmr":
        lab = 1   # MSI
    else:
        continue  # NA, missing, or ambiguous — skip
    keep_samples.append(sid)
    labels.append(lab)

expr = expr_wide.loc[keep_samples].astype(np.float32)
labels = np.asarray(labels, dtype=np.int64)
sample_ids = np.asarray(expr.index.astype(str))
gene_names = np.asarray(expr.columns.astype(str))

print("labeled samples:", len(labels))
print("  MSI (dMMR):", int(labels.sum()))
print("  MSS (pMMR):", int((1-labels).sum()))

# Quick check: print any unexpected values found
all_vals = gse.phenotype_data[MMR_COL].dropna().unique()
print("\nAll MMR values seen in metadata:", all_vals)

labeled samples: 536
  MSI (dMMR): 77
  MSS (pMMR): 459

All MMR values seen in metadata: ['pMMR' 'dMMR' 'N/A']


In [10]:
# GSE39582 is microarray intensity data, not RNA-seq counts.
# Use microarray-safe preprocessing: z-score each gene across samples.
bulk_expr = expr.values.astype(np.float32)
mu = np.nanmean(bulk_expr, axis=0, keepdims=True)
sd = np.nanstd(bulk_expr, axis=0, keepdims=True)
sd = np.where(sd < 1e-6, 1.0, sd)
bulk_expr = ((bulk_expr - mu) / sd).astype(np.float32)
bulk_genes = gene_names
bulk_labels = labels
bulk_sample_ids = sample_ids
print("bulk_expr", bulk_expr.shape, "range", float(np.nanmin(bulk_expr)), float(np.nanmax(bulk_expr)))


bulk_expr (536, 54675) range -10.813822746276855 15.06829833984375


In [11]:
# Save
np.save(DATA_DIR / "bulk_expr.npy", bulk_expr)
np.save(DATA_DIR / "bulk_genes.npy", bulk_genes)
np.save(DATA_DIR / "bulk_labels.npy", bulk_labels)
np.save(DATA_DIR / "bulk_sample_ids.npy", bulk_sample_ids)
print("Saved to", DATA_DIR)


Saved to C:\Users\krith\OneDrive - The University of Memphis\Neural Network\Project\data
